# FHI SME Challenge — Modelling and Evaluation

Documents the full modelling pipeline: ordinal binary decomposition with
LightGBM + XGBoost + CatBoost ensemble, joint country×target CV stratification,
and OOF-based threshold tuning.

**Best results:**
- OOF weighted F1: 0.8791
- Public leaderboard: 0.8857
- Private leaderboard: 0.8852 (rank 83)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
from pathlib import Path
from sklearn.metrics import (
    f1_score, confusion_matrix, classification_report
)
warnings.filterwarnings("ignore")

PROCESSED_DIR = Path("../data/processed")
MODELS_DIR     = Path("../models")

# Load processed features
train = pd.read_csv(PROCESSED_DIR / "train_features.csv")
print(f"Train features shape: {train.shape}")
print(f"\nTarget distribution:")
print(train["Target"].value_counts().sort_index()
      .rename({0: "Low", 1: "Medium", 2: "High"}))

## 1. Modelling Approach

### Why ordinal decomposition?

The target (Low / Medium / High) is ordinal — there is a meaningful direction.
Standard multiclass treats all misclassifications equally. Ordinal decomposition
enforces the ordering:

- **Task A**: P(Target ≥ Medium) — separates Low from {Medium, High}
- **Task B**: P(Target = High) — separates High from {Low, Medium}

Decision rule at inference:
```
if P_B ≥ tB  → predict High
elif P_A ≥ tA → predict Medium  
else          → predict Low
```

This eliminates catastrophic Low↔High confusions by construction.

In [ ]:
# Load saved artifacts
with open(MODELS_DIR / "ordinal_artifacts.pkl", "rb") as f:
    artifacts = pickle.load(f)

print("Artifacts loaded:")
print(f"  OOF weighted F1:  {artifacts['oof_f1']:.4f}")
print(f"  Ensemble weights: LGB={artifacts['best_weights'][0]:.2f}  "
      f"XGB={artifacts['best_weights'][1]:.2f}  "
      f"CAT={artifacts['best_weights'][2]:.2f}")
print(f"  tA (>=Medium):    {artifacts['best_tA']:.3f}")
print(f"  tB (==High):      {artifacts['best_tB']:.3f}")
print(f"  Feature count:    {len(artifacts['feature_cols'])}")

## 2. OOF Predictions and Threshold Tuning

In [ ]:
# Reconstruct OOF predictions from saved arrays
w_lgb, w_xgb, w_cat = artifacts["best_weights"]

oof_A = (w_lgb * artifacts["oof_A_lgb"] +
         w_xgb * artifacts["oof_A_xgb"] +
         w_cat * artifacts["oof_A_cat"])

oof_B = (w_lgb * artifacts["oof_B_lgb"] +
         w_xgb * artifacts["oof_B_xgb"] +
         w_cat * artifacts["oof_B_cat"])

y_true = train["Target"].values

def ordinal_predict(p_A, p_B, tA, tB):
    preds = np.zeros(len(p_A), dtype=int)
    preds[p_A >= tA] = 1
    preds[p_B >= tB] = 2
    return preds

final_preds = ordinal_predict(oof_A, oof_B, artifacts["best_tA"], artifacts["best_tB"])
oof_f1 = f1_score(y_true, final_preds, average="weighted")
print(f"OOF weighted F1: {oof_f1:.4f}")

In [ ]:
# Threshold search visualisation — show F1 surface over tB grid
from sklearn.metrics import f1_score as sk_f1

tB_grid = np.linspace(0.1, 0.9, 50)
tA_fixed = artifacts["best_tA"]
f1_scores = []
for tB in tB_grid:
    preds = ordinal_predict(oof_A, oof_B, tA_fixed, tB)
    f1_scores.append(sk_f1(y_true, preds, average="weighted"))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(tB_grid, f1_scores, color="steelblue", linewidth=2)
ax.axvline(artifacts["best_tB"], color="coral", linestyle="--",
           label=f"Best tB = {artifacts['best_tB']:.3f}")
ax.set_xlabel("tB threshold (High class)")
ax.set_ylabel("OOF weighted F1")
ax.set_title(f"F1 vs High threshold (tA fixed at {tA_fixed:.3f})")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("../outputs/figures/threshold_curve.png", dpi=120, bbox_inches="tight")
plt.show()

## 3. Confusion Matrix and Error Analysis

In [ ]:
# Overall confusion matrix
cm = confusion_matrix(y_true, final_preds, labels=[0, 1, 2])
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["Low", "Medium", "High"],
            yticklabels=["Low", "Medium", "High"])
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"OOF Confusion Matrix (weighted F1={oof_f1:.4f})")
plt.tight_layout()
plt.savefig("../outputs/figures/confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Error breakdown
off_diag = cm.copy()
np.fill_diagonal(off_diag, 0)
total_err = off_diag.sum()
low_med   = off_diag[0,1] + off_diag[1,0]
med_high  = off_diag[1,2] + off_diag[2,1]
low_high  = off_diag[0,2] + off_diag[2,0]

print("Error breakdown:")
print(f"  Low↔Medium:  {low_med:>5} ({low_med/total_err*100:.1f}%)")
print(f"  Medium↔High: {med_high:>5} ({med_high/total_err*100:.1f}%)")
print(f"  Low↔High:    {low_high:>5} ({low_high/total_err*100:.1f}%)")
print()
print("Per-class report:")
print(classification_report(y_true, final_preds,
      target_names=["Low", "Medium", "High"], digits=4))

## 4. Per-Country Performance

In [ ]:
# Per-country diagnostics
country_arr = train["country"].values
COUNTRY_NAMES = {0: "Lesotho", 1: "Zim/Malawi", 3: "Eswatini"}

results = []
for val, name in sorted(COUNTRY_NAMES.items()):
    mask = country_arr == val
    if mask.sum() == 0:
        continue
    y_c = y_true[mask]
    p_c = final_preds[mask]
    f1_c = f1_score(y_c, p_c, average="weighted", zero_division=0)
    results.append({
        "Country": name,
        "N": mask.sum(),
        "High_true": (y_c == 2).sum(),
        "High_pred": (p_c == 2).sum(),
        "High_rate": f"{(y_c==2).mean():.1%}",
        "WF1": f"{f1_c:.4f}"
    })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

In [ ]:
# Per-country confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (val, name) in zip(axes, sorted(COUNTRY_NAMES.items())):
    mask = country_arr == val
    if mask.sum() == 0:
        continue
    cm_c = confusion_matrix(y_true[mask], final_preds[mask], labels=[0,1,2])
    f1_c = f1_score(y_true[mask], final_preds[mask], average="weighted", zero_division=0)
    sns.heatmap(cm_c, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Low","Med","High"],
                yticklabels=["Low","Med","High"])
    ax.set_title(f"{name}\nWF1={f1_c:.4f}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
plt.suptitle("Per-Country Confusion Matrices", y=1.02)
plt.tight_layout()
plt.savefig("../outputs/figures/per_country_confusion.png", dpi=120, bbox_inches="tight")
plt.show()

## 5. Key Findings and Lessons Learned

### What worked
- **Metric alignment**: switching from macro to weighted F1 gave +0.037 OOF
- **Product NaN → 0**: imputing financial product NaN as "Never had" gave +0.040 OOF — largest single gain
- **Ordinal decomposition**: two binary models instead of multiclass gave +0.002 OOF
- **Country×target stratification**: tighter CV alignment with leaderboard
- **Unicode NFKC normalisation**: consolidated scattered "Don't know" variants, changed ensemble weights from LGB=0.90/CAT=0.00 to LGB=0.40/CAT=0.50

### What did not work
- Country separation (hurt overall due to Lesotho's 20% weight)
- WOE encoding (no improvement)
- Optuna hyperparameter tuning (no meaningful gain)
- Pseudo-labelling (marginal effect)
- Sentinel separation (redundant with country column)

### The Lesotho problem
Lesotho uses a different survey instrument — 12 columns are structurally absent.
Only 6 High cases in 1,944 rows (0.3%). This creates a hard ceiling:
- Lesotho OOF is ~0.72 under every approach
- Lesotho is 20% of training data
- Every +0.01 gain in other countries is partially neutralised

### Leaderboard stability
Our consistent +0.010 gap (OOF → public) held across every experiment.
This predicted the private leaderboard correctly — our public score dropped
only 0.0044 on the private reveal, smaller than competitors who chased
the public leaderboard through repeated probing.

In [ ]:
# Score progression table
scores = [
    ("Starter baseline (macro F1, class weights)", None, 0.8514),
    ("Switch to weighted F1, remove class weights", 0.8462, 0.8828),
    ("Product NaN → 0 imputation",                  0.8071, 0.8865),
    ("Ordinal binary decomposition",                0.8791, 0.8895),
    ("Unicode NFKC normalisation",                  0.8791, 0.8857),
]

df_scores = pd.DataFrame(scores, columns=["Experiment", "OOF F1", "Public F1"])
print(df_scores.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(scores))
public = [s[2] for s in scores]
oof    = [s[1] if s[1] else s[2]-0.01 for s in scores]
ax.plot(x, public, "o-", color="steelblue", label="Public LB", linewidth=2, markersize=7)
ax.plot(x, oof,    "s--", color="coral", label="OOF", linewidth=2, markersize=7)
ax.set_xticks(x)
ax.set_xticklabels([s[0] for s in scores], rotation=20, ha="right", fontsize=8)
ax.set_ylabel("Weighted F1")
ax.set_title("Score Progression Across Experiments")
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim(0.83, 0.91)
plt.tight_layout()
plt.savefig("../outputs/figures/score_progression.png", dpi=120, bbox_inches="tight")
plt.show()